# Computer Vision Assignment 2
## Chest X-Ray Image Classification using TensorFlow and Keras

**Student Name:** Joel George  
**Student Number:** B00118121  
**Module:** Computer Vision  

### Goal
Maximise classification accuracy, precision, and recall on the chest X-ray dataset.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd

print("TensorFlow version:", tf.__version__)

In [ ]:
# Path to the training folder
# Change this if your folder is in a different place
train_dir = r"C:\Assignment 2 - B00118121\chest_xray\chest_xray\train"

# Path to the test folder
test_dir  = r"C:\Assignment 2 - B00118121\chest_xray\chest_xray\test"

# Resize all images to 128x128 before training
# This makes the input size fixed for the neural network
img_height = 128
img_width = 128

# Number of images processed at once during training
batch_size = 16

# Seed makes the dataset split repeatable
# This means you get the same training/validation split each time
seed = 123

# Number of training passes over the full dataset
epochs = 15

In [ ]:
# Create the training dataset from the train folder
# validation_split=0.2 means 20% of the training folder will be used for validation
# subset="training" means this dataset is the 80% training part
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="int"   # labels are integers like 0, 1, 2
)

# Create the validation dataset from the same train folder
# subset="validation" means this is the remaining 20%
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="int"
)

# Create the separate test dataset from the test folder
# shuffle=False is important so evaluation order stays fixed
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False,
    label_mode="int"
)

# Save the class names from the folder names
# For your dataset this should be something like:
# ['BACTERIAL', 'NORMAL', 'VIRAL']
class_names = train_ds.class_names

# Print class names to check they loaded correctly
print("Classes:", class_names)

In [ ]:
# Create a figure window big enough to show 9 images
plt.figure(figsize=(10, 10))

# Take just one batch from the training dataset
for images, labels in train_ds.take(1):
    
    # Loop through the first 9 images in the batch
    for i in range(9):
        # Create a 3x3 grid of plots
        ax = plt.subplot(3, 3, i + 1)
        
        # Show the image
        plt.imshow(images[i].numpy().astype("uint8"))
        
        # Show the class name as the title
        plt.title(class_names[labels[i].numpy()])
        
        # Hide axis numbers for cleaner display
        plt.axis("off")

# Make spacing look better
plt.tight_layout()

# Display the images
plt.show()

In [ ]:
# Create a dictionary to count how many images are in each class
# Start all class counts at 0
train_counts = {name: 0 for name in class_names}

# Go through every label in the training dataset one by one
for _, labels in train_ds.unbatch():
    # Convert the label number to the class name and increase its count
    train_counts[class_names[int(labels.numpy())]] += 1

# Print the class counts
print("Training set class distribution:")
print(train_counts)

# Plot the class counts as a bar chart
plt.figure(figsize=(6,4))
plt.bar(train_counts.keys(), train_counts.values())
plt.title("Training Set Class Distribution")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
# Create an empty list to store all training labels
y_train_all = []

# Go through every training label in the dataset
for _, labels in train_ds.unbatch():
    # Add each label as a normal integer
    y_train_all.append(int(labels.numpy()))

# Convert the list to a NumPy array
y_train_all = np.array(y_train_all)

# Compute balanced class weights
# This gives bigger weights to smaller classes
# So mistakes on under-represented classes matter more during training
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_all),
    y=y_train_all
)

# Turn the array into a dictionary format Keras expects
# Example: {0: 1.2, 1: 0.8, 2: 1.5}
class_weights = {i: class_weights_array[i] for i in range(len(class_weights_array))}

# Print the weights
print("Class weights:", class_weights)

In [ ]:
# Create an empty list to store all training labels
y_train_all = []

# Go through every training label in the dataset
for _, labels in train_ds.unbatch():
    # Add each label as a normal integer
    y_train_all.append(int(labels.numpy()))

# Convert the list to a NumPy array
y_train_all = np.array(y_train_all)

# Compute balanced class weights
# This gives bigger weights to smaller classes
# So mistakes on under-represented classes matter more during training
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_all),
    y=y_train_all
)

# Turn the array into a dictionary format Keras expects
# Example: {0: 1.2, 1: 0.8, 2: 1.5}
class_weights = {i: class_weights_array[i] for i in range(len(class_weights_array))}

# Print the weights
print("Class weights:", class_weights)

In [ ]:
# This function builds our CNN model
def make_cnn_model():
    model = models.Sequential([
        
        # Input layer: images are 128x128 with 3 colour channels
        layers.Input(shape=(img_height, img_width, 3)),
        
        # Apply data augmentation only during training
        data_augmentation,
        
        # Scale pixel values from 0-255 down to 0-1
        layers.Rescaling(1./255),

        # First convolution block
        # Learns simple features like edges and textures
        layers.Conv2D(32, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Second convolution block
        # Learns more complex patterns
        layers.Conv2D(64, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Third convolution block
        # Learns even higher-level image features
        layers.Conv2D(128, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # GlobalAveragePooling2D replaces Flatten
        # It reduces the number of parameters and can reduce overfitting
        layers.GlobalAveragePooling2D(),

        # Dense layer helps combine learned features for classification
        layers.Dense(128, activation='relu'),
        
        # Dropout randomly turns off some neurons during training
        # This helps prevent overfitting
        layers.Dropout(0.3),

        # Final output layer
        # Number of outputs = number of classes
        # Softmax gives class probabilities
        layers.Dense(len(class_names), activation='softmax')
    ])

    # Compile the model
    # Adam = optimiser
    # sparse_categorical_crossentropy = correct loss for integer class labels
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build the model
cnn_model = make_cnn_model()

# Show model structure
cnn_model.summary()

In [ ]:
# EarlyStopping watches validation loss
# If it does not improve for 5 epochs, training stops early
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# ModelCheckpoint saves the best model file during training
checkpoint = ModelCheckpoint(
    "best_cnn_model.keras",
    save_best_only=True
)

# Record start time before training
start_time = time.time()

# Train the CNN model
cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,                  # try up to 20 epochs
    callbacks=[early_stop, checkpoint],
    class_weight=class_weights  # use class weights to help imbalance
)

# Calculate total training time
cnn_train_time = time.time() - start_time

# Print training time
print(f"CNN training time: {cnn_train_time:.2f} seconds")

In [ ]:
# EarlyStopping watches validation loss
# If it does not improve for 5 epochs, training stops early
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# ModelCheckpoint saves the best model file during training
checkpoint = ModelCheckpoint(
    "best_cnn_model.keras",
    save_best_only=True
)

# Record start time before training
start_time = time.time()

# Train the CNN model
cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,                  # try up to 20 epochs
    callbacks=[early_stop, checkpoint],
    class_weight=class_weights  # use class weights to help imbalance
)

# Calculate total training time
cnn_train_time = time.time() - start_time

# Print training time
print(f"CNN training time: {cnn_train_time:.2f} seconds")

In [ ]:
# Evaluate the trained CNN on the separate test set
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(test_ds)

# Print final test accuracy
print("CNN Test Accuracy:", cnn_test_acc)

In [ ]:
# Store true labels and predicted labels
y_true = []
y_pred = []

# Go through every batch in the test dataset
for images, labels in test_ds:
    
    # Get predicted probabilities from the model
    preds = cnn_model.predict(images, verbose=0)
    
    # Add the real labels to the list
    y_true.extend(labels.numpy())
    
    # Convert predicted probabilities to class indexes
    y_pred.extend(np.argmax(preds, axis=1))

# Print the classification report
# This includes precision, recall and F1-score for each class
print("=== CNN Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Display it nicely
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues")

# Add title
plt.title("CNN Confusion Matrix")

# Show it
plt.show()

In [ ]:
# This function builds a transfer learning model
def make_transfer_model():
    
    # Load MobileNetV2 without its original final classifier
    # weights='imagenet' means it starts with knowledge learned from a huge dataset
    base_model = MobileNetV2(
        input_shape=(img_height, img_width, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze the base model so its pre-trained weights do not change at first
    base_model.trainable = False

    # Define a new input layer
    inputs = keras.Input(shape=(img_height, img_width, 3))
    
    # Apply data augmentation
    x = data_augmentation(inputs)
    
    # Rescale image pixels to 0-1
    x = layers.Rescaling(1./255)(x)
    
    # Pass images through the pre-trained MobileNetV2
    x = base_model(x, training=False)
    
    # Reduce feature maps to one vector per image
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dropout for regularisation
    x = layers.Dropout(0.3)(x)
    
    # New classifier for our 3 chest X-ray classes
    outputs = layers.Dense(len(class_names), activation='softmax')(x)

    # Build the final model
    model = keras.Model(inputs, outputs)

    # Compile the model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build the transfer learning model
transfer_model = make_transfer_model()

# Show model structure
transfer_model.summary()

In [ ]:
# Early stopping for transfer learning model
early_stop_tl = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Save the best transfer model
checkpoint_tl = ModelCheckpoint(
    "best_transfer_model.keras",
    save_best_only=True
)

# Start timer
start_time = time.time()

# Train transfer learning model
transfer_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop_tl, checkpoint_tl],
    class_weight=class_weights
)

# Measure total training time
transfer_train_time = time.time() - start_time

# Print training time
print(f"Transfer model training time: {transfer_train_time:.2f} seconds")

In [ ]:
# Create figure for two graphs
plt.figure(figsize=(12,4))

# ---- Accuracy graph ----
plt.subplot(1,2,1)
plt.plot(transfer_history.history['accuracy'], label='Train Accuracy')
plt.plot(transfer_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Transfer Learning Accuracy')
plt.legend()

# ---- Loss graph ----
plt.subplot(1,2,2)
plt.plot(transfer_history.history['loss'], label='Train Loss')
plt.plot(transfer_history.history['val_loss'], label='Validation Loss')
plt.title('Transfer Learning Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate transfer model on the test set
transfer_test_loss, transfer_test_acc = transfer_model.evaluate(test_ds)

# Print test accuracy
print("Transfer Learning Test Accuracy:", transfer_test_acc)

In [ ]:
# Store true labels and predictions for transfer model
y_true_tl = []
y_pred_tl = []

# Loop through the test dataset
for images, labels in test_ds:
    
    # Predict class probabilities
    preds = transfer_model.predict(images, verbose=0)
    
    # Save true labels
    y_true_tl.extend(labels.numpy())
    
    # Save predicted class indexes
    y_pred_tl.extend(np.argmax(preds, axis=1))

# Print classification report
print("=== Transfer Learning Classification Report ===")
print(classification_report(y_true_tl, y_pred_tl, target_names=class_names))

# Create confusion matrix
cm_tl = confusion_matrix(y_true_tl, y_pred_tl)

# Display it
disp = ConfusionMatrixDisplay(confusion_matrix=cm_tl, display_labels=class_names)
disp.plot(cmap="Greens")
plt.title("Transfer Learning Confusion Matrix")
plt.show()